In [1]:
# !pip install transformers datasets sentencepiece accelerate -q

In [2]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)

In [3]:
BASE_MODEL  = "csebuetnlp/banglat5"
DATA_CSV    = "resume_summarization_dataset.csv"
OUTPUT_DIR  = "bangla_resume_summarizer_banglat5"

MAX_INPUT   = 512
MAX_TARGET  = 128
BATCH_SIZE  = 4
EPOCHS      = 5
LR          = 5e-4   # safe with fp16=False

In [4]:
df = pd.read_csv(DATA_CSV).dropna(subset=["Input_Text", "Reference_Summary"])
df = df[["Input_Text", "Reference_Summary"]].reset_index(drop=True)

print(f"Total pairs: {len(df)}")

split    = int(len(df) * 0.8)
train_df = df.iloc[:split].reset_index(drop=True)
val_df   = df.iloc[split:].reset_index(drop=True)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}")

train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)

Total pairs: 396
Train: 316  |  Val: 80


In [5]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)

def preprocess(batch):
    inputs  = [t.strip() for t in batch["Input_Text"]]
    targets = [s.strip() for s in batch["Reference_Summary"]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT,
        truncation=True,
        padding=False,
    )

    # FIX: use text_target (not as_target_tokenizer — deprecated)
    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET,
        truncation=True,
        padding=False,
    )

    # Mask padding tokens so they don't contribute to loss
    labels["input_ids"] = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_ds.map(preprocess, batched=True,
                                remove_columns=["Input_Text", "Reference_Summary"])
tokenized_val   = val_ds.map(preprocess,   batched=True,
                                remove_columns=["Input_Text", "Reference_Summary"])

# Quick sanity check — make sure labels are NOT all -100
sample_labels = tokenized_train[0]["labels"]
valid_tokens  = [t for t in sample_labels if t != -100]
print(f"Sample label token count: {len(valid_tokens)} valid / {len(sample_labels)} total")
assert len(valid_tokens) > 0, "All labels are -100 — preprocessing is broken!"
print("Label check passed.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

Map:   0%|          | 0/316 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Sample label token count: 35 valid / 35 total
Label check passed.


In [6]:
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
print(f"Model parameters: {model.num_parameters():,}")

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model parameters: 247,577,856


In [7]:
args = Seq2SeqTrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate               = LR,
    warmup_steps                = 50,
    weight_decay                = 0.01,
    predict_with_generate       = True,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    fp16                        = False,       # FIX: BanglaT5 overflows with fp16
    logging_steps               = 10,
    save_total_limit            = 2,
    report_to                   = "none",
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

In [8]:
trainer = Seq2SeqTrainer(
    model            = model,
    args             = args,
    train_dataset    = tokenized_train,
    eval_dataset     = tokenized_val,
    processing_class = tokenizer,
    data_collator    = data_collator,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting BanglaT5 fine-tuning...")
trainer.train()

Starting BanglaT5 fine-tuning...


Epoch,Training Loss,Validation Loss
1,3.565442,2.297078
2,2.173856,1.685621
3,1.578585,1.517105
4,1.499516,1.446138
5,1.420519,1.440408


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=395, training_loss=3.2024091527431824, metrics={'train_runtime': 328.7802, 'train_samples_per_second': 4.806, 'train_steps_per_second': 1.201, 'total_flos': 222738727550976.0, 'train_loss': 3.2024091527431824, 'epoch': 5.0})

In [11]:
import os
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

if os.path.exists(OUTPUT_DIR):
    files = os.listdir(OUTPUT_DIR)
    print(f"Model saved to '{OUTPUT_DIR}/'")
    print(f"Files: {files}")
else:
    print("ERROR: Model folder still not found — something went wrong.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to 'bangla_resume_summarizer_banglat5/'
Files: ['model.safetensors', 'tokenizer_config.json', 'config.json', 'checkpoint-316', 'tokenizer.json', 'checkpoint-395', 'generation_config.json', 'training_args.bin']


In [12]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

DRIVE_SAVE_PATH = "/content/drive/MyDrive/bangla_resume_summarizer_banglat5"
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

for item in os.listdir(OUTPUT_DIR):
    if item.startswith("checkpoint-"):
        print(f"  Skipped: {item}")
        continue
    src  = os.path.join(OUTPUT_DIR, item)
    dest = os.path.join(DRIVE_SAVE_PATH, item)
    if os.path.isdir(src):
        shutil.copytree(src, dest, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dest)
    print(f"  Copied: {item}")

print(f"\nSaved to Google Drive: '{DRIVE_SAVE_PATH}'")

Mounted at /content/drive
  Copied: model.safetensors
  Copied: tokenizer_config.json
  Copied: config.json
  Skipped: checkpoint-316
  Copied: tokenizer.json
  Skipped: checkpoint-395
  Copied: generation_config.json
  Copied: training_args.bin

Saved to Google Drive: '/content/drive/MyDrive/bangla_resume_summarizer_banglat5'


In [10]:
# ── Cell 11: Sanity check ─────────────────────────────────────────────────────
print("\n--- Sanity Check (first 3 val samples) ---")
model.eval()
for i in range(min(3, len(val_df))):
    input_text = val_df.iloc[i]["Input_Text"]
    inputs = tokenizer(input_text, return_tensors="pt",
                       truncation=True, max_length=MAX_INPUT)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}  # FIX
    with torch.no_grad():
        out = model.generate(inputs["input_ids"], max_length=128,
                             num_beams=4, early_stopping=True)
    pred = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"\nInput    : {val_df.iloc[i]['Input_Text'][:80]}...")
    print(f"Reference: {val_df.iloc[i]['Reference_Summary']}")
    print(f"Generated: {pred}")


--- Sanity Check (first 3 val samples) ---

Input    : স্বাস্থ্যসেবা খাতে ডেটা অ্যানালিস্ট হিসেবে কাজ করা, যেখানে ক্লিনিকাল ডেটা বিশ্লে...
Reference: স্বাস্থ্যসেবা ক্ষেত্রে ডেটা অ্যানালিস্ট হিসেবে কাজ করতে ইচ্ছুক। ক্লিনিকাল ডেটা বিশ্লেষণের মাধ্যমে রোগীর সেবার মান উন্নত করা এবং হাসপাতালের কার্যকারিতা বৃদ্ধি করা লক্ষ্য।
Generated: প্রার্থী স্বাস্থ্যসেবা খাতে ডেটা অ্যানালিস্ট হিসেবে কাজ করে ক্লিনিকাল ডেটা বিশ্লেষণের মাধ্যমে রোগীর সেবার মান উন্নত করতে ইচ্ছুক। হাসপাতালের কার্যকারিতা বৃদ্ধি করে রোগীর সেবার মান উন্নত করা সম্ভব।

Input    : ক্লিনিকাল ডেটা অ্যানালিস্ট । সেপ্টেম্বর ২০২১ - বর্তমান । রোগীর তথ্য এবং ক্লিনিকা...
Reference: সেপ্টেম্বর ২০২১ থেকে বর্তমান পর্যন্ত ক্লিনিকাল ডেটা অ্যানালিস্ট হিসেবে রোগীর তথ্য ও ক্লিনিকাল ট্রায়াল ডেটা বিশ্লেষণ করেন। ডেটা প্রাইভেসি ও কমপ্লায়েন্স বজায় রেখে হাসপাতালের রিসোর্স অ্যালোকেশন রিপোর্ট প্রস্তুত করেন।
Generated: প্রার্থী ক্লিনিকাল ডেটা অ্যানালিস্ট হিসেবে সেপ্টেম্বর ২০২১ থেকে ক্লিনিক্যাল ডেটা অ্যানালিস্ট হিসেবে কাজ করছেন, যেখানে রোগীর তথ্য ও ক্লিনি